# Sensitivity Analysis for FLOWGEM

In [ ]:
import importlib
import subprocess

def install_if_missing(package, import_name=None):
    name = import_name or package
    if importlib.util.find_spec(name) is None:
        subprocess.run(["pip", "install", package], check=True)

In [ ]:
import os
import sys
import torch
import time

from pathlib import Path

import numpy as np
from matplotlib import pyplot as plt
from scipy.spatial.distance import cdist, pdist
import pandas as pd
install_if_missing("pyreadr")
import pyreadr
from flowgem import sample_flowgem

install_if_missing("hyperimpute")
from hyperimpute.plugins.imputers import Imputers

install_if_missing("pot")
import ot

base_path = "./"

In [ ]:
# Define file paths
save_dir = f"{base_path}/results/sensitivity_analysis"
log_file = os.path.join(save_dir, 'parameter_log.csv')
impute_dir = f"{save_dir}/imputations"
metric_dir = f"{save_dir}/metrics"

## Set parameters

In [ ]:
datasets = ["housing", "forest", "concrete"]
n_runs = 5

defaults = {"init": "mice", "sigma_mul": 1.0, "eta": 0.01, "T": 1000}

fixed_T = 5000

sigma_mul_values = np.array([0.25, 0.5, 1.0, 2.0, 4.0])
eta_values = np.array([0.25, 0.5, 1.0, 2.0, 4.0]) * defaults["eta"]

## Define init function and evaluation metrics

In [ ]:
def impute_bootstrap_per_col(X, M):
    n, d = X.shape
    for j in range(d):
        observed_mask = M[:, j] == 1
        missing_mask = M[:, j] == 0

        observed_values = X[observed_mask, j]
        if observed_values.numel() == 0:
            raise ValueError(f"Value of column {j} is always missing. Need to be observed at least once.")

        num_missing = missing_mask.sum()
        if num_missing > 0:
            rand_idx = torch.randint(0, observed_values.shape[0], (num_missing,), device=X.device)
            X[missing_mask, j] = observed_values[rand_idx]


def energy_distance(X, Y, scale):
    if scale:
        X_np = np.array(X)
        Y_np = np.array(Y)
        center = np.nanmean(X_np, axis=0)
        scl = np.nanstd(X_np, axis=0, ddof=1)

        X = (X_np - center) / scl

        # Scale imputed data using original data's mean and std
        Y = (Y_np - center) / scl

    XY = cdist(X, Y)
    XX = cdist(X, X)
    YY = cdist(Y, Y)
    return (2 * XY.mean() - XX.mean() - YY.mean()) * X.shape[0] / 2

def energy_distance_faster(X, Y, scale):
    if torch.max(torch.abs(X)).item() > 1e100 and torch.max(torch.abs(Y)).item() < 1e100:
        return torch.inf
    if scale:
        X_np = np.array(X)
        Y_np = np.array(Y)
        center = np.nanmean(X_np, axis=0)
        scl = np.nanstd(X_np, axis=0, ddof=1)

        X = (X_np - center) / scl

        # Scale imputed data using original data's mean and std
        Y = (Y_np - center) / scl

    n = X.shape[0]
    xx_mean = pdist(X).sum() * 2 / (n * n)   # divide by n^2 to match cdist mean
    yy_mean = pdist(Y).sum() * 2 / (n * n)   # same for Y
    xy_mean = cdist(X, Y).mean()

    return (2 * xy_mean - xx_mean - yy_mean) * n / 2

def energy_distance_fixed_X(X, XX_mean, Y):
    XY_mean = cdist(X, Y).mean()
    n = len(Y)
    YY_mean = pdist(Y).mean() * (n - 1) / n
    return 2 * XY_mean - XX_mean - YY_mean


def W2(X, Y, scale):
    if torch.max(torch.abs(X)).item() > 1e100 and torch.max(torch.abs(Y)).item() < 1e100:
        return torch.inf
    if scale:
        X_np = np.array(X)
        Y_np = np.array(Y)
        center = np.nanmean(X_np, axis=0)
        scl = np.nanstd(X_np, axis=0, ddof=1)

        X = (X_np - center) / scl

        # Scale imputed data using original data's mean and std
        Y = (Y_np - center) / scl
    else:
        X = np.array(X)
        Y = np.array(Y)

    dists = ((X[:, None] - Y) ** 2).sum(2) / 2.
    n = X.shape[0]
    return ot.emd2(np.ones(n) / n, np.ones(n) / n, dists, numItermax=1_000_000)

def median_heuristic_sigma(X, Y, eps=1e-12):
    X = np.asarray(X)
    Y = np.asarray(Y)

    Z = np.concatenate([X, Y], axis=0)  # (n+m, d)

    # pairwise squared distances
    Z = Z.astype(np.float64, copy=False)
    Z_norm = (Z**2).sum(axis=1, keepdims=True)          # (N,1)
    dist2 = Z_norm + Z_norm.T - 2.0 * (Z @ Z.T)       # (N,N)

    # off-diagonal entries only
    N = dist2.shape[0]
    off_diag = dist2[~np.eye(N, dtype=bool)]

    sigma2 = np.median(off_diag)
    sigma2 = max(sigma2, eps)
    return np.sqrt(sigma2)

def MMD(X, Y, scale, sigma=None):
    if scale:
        X_np = np.array(X)
        Y_np = np.array(Y)
        center = np.nanmean(X_np, axis=0)
        scl = np.nanstd(X_np, axis=0, ddof=1)

        X = (X_np - center) / scl

        # Scale imputed data using original data's mean and std
        Y = (Y_np - center) / scl
    else:
        X = np.array(X)
        Y = np.array(Y)

    if sigma is None:
        sigma = median_heuristic_sigma(X, Y)

    sigma2 = sigma ** 2

    def rbf(a, b):
        a2 = (a ** 2).sum(axis=1, keepdims=True)   # (n,1)
        b2 = (b ** 2).sum(axis=1, keepdims=True).T # (1,m)
        dist2 = a2 + b2 - 2.0 * (a @ b.T)          # (n,m)
        return np.exp(-dist2 / (2.0 * sigma2))

    Kxx = rbf(X, X)
    Kyy = rbf(Y, Y)
    Kxy = rbf(X, Y)

    return Kxx.mean() + Kyy.mean() - 2 * Kxy.mean()


## Generate samples and evaluate merics

In [ ]:
param_configs = []

# Sweep sigma, eta fixed at default
for sigma_mul in sigma_mul_values:
    param_configs.append({"init": defaults["init"], "sigma_mul": float(sigma_mul), "eta": defaults["eta"], "T": defaults["T"]})

# Sweep eta, sigma fixed at default
for eta in eta_values:
    param_configs.append({"init": defaults["init"], "sigma_mul": defaults["sigma_mul"], "eta": float(eta), "T": (int(10/eta) if fixed_T == 0 else fixed_T)})

for param_vals in param_configs:
    print(param_vals)

    #=======================#
    #=== LOG FILE AND ID ===#
    #=======================#
    # Check if the log file exists and read it
    if os.path.exists(log_file):
        param_log = pd.read_csv(log_file)
    else:
        param_log = pd.DataFrame(columns=['ID', 'init', 'sigma_mul', 'eta', 'T'])

    # Determine the next ID
    this_id = 1 if param_log.empty else param_log['ID'].max() + 1

    # Check if this setup already exists and keep same ID
    query_str = " & ".join([f"{k} == {repr(v)}" for k, v in param_vals.items()])
    matching = param_log.query(query_str)
    if not matching.empty:
        this_id = matching['ID'].iloc[0]
    else:
        new_row = {'ID': this_id, **param_vals}
        param_log = pd.concat([param_log, pd.DataFrame([new_row])], ignore_index=True)
        param_log.to_csv(log_file, index=False)
    print("ID:", this_id)

    #=================================#
    #=== GENERATION AND EVALUATION ===#
    #=================================#
    for dataset in datasets:
        needed_runs = [k for k in range(n_runs) if not os.path.exists(f"{metric_dir}/metrics_{this_id}_{dataset}_{k}.pt")]
        if needed_runs == 0:
            continue

        Xstar_df = pyreadr.read_r(f"{base_path}/datasets/split_test/test.{dataset}.RDS")[None]
        X_miss_df = pyreadr.read_r(f"{base_path}/datasets/split_amputed/mar.{dataset}.RDS")[None]

        for run in needed_runs:
            #============= GENERATION ==============#
            if os.path.exists(f"{impute_dir}/imputation_{this_id}_{dataset}_{run}.pt"):
                imp_save = torch.load(f"{impute_dir}/imputation_{this_id}_{dataset}_{run}.pt", weights_only=False)
            else:
                torch.manual_seed(run)
                np.random.seed(run)

                ## M=1-M in the paper
                M_np = (X_miss_df.notna()).astype(int).values
                M_tensor = torch.tensor(M_np, dtype=torch.float64)
                X0_tensor = torch.tensor(X_miss_df.values, dtype=torch.float64)

                if param_vals["init"] == "ColBT":
                    # bootstrap imputation
                    impute_bootstrap_per_col(X0_tensor, M_tensor) # modifies X0 in-place

                    # standardize X0 using observed values only
                    Xmiss_tensor = torch.tensor(X_miss_df.values, dtype=torch.float64)
                    col_std = torch.nanmean((Xmiss_tensor - torch.nanmean(Xmiss_tensor, dim=0)) ** 2, dim=0) ** 0.5
                    col_mean = torch.nanmean(Xmiss_tensor, dim=0)

                elif param_vals["init"] == "mice":
                    # Use mice imputation with 10 iterations as X0
                    Xmiss_tensor = torch.tensor(X_miss_df.values, dtype=torch.float64)

                    col_std = torch.nanmean((Xmiss_tensor - torch.nanmean(Xmiss_tensor, dim=0)) ** 2, dim=0) ** 0.5
                    col_mean = torch.nanmean(Xmiss_tensor, dim=0)

                    imputer = Imputers().get("mice", max_iter=10, random_state=run)
                    df = imputer.fit_transform(X_miss_df)
                    X0_np = df.values
                    X0_tensor = torch.tensor(X0_np, dtype=torch.float64)

                    print(np.isnan(X0_np).any())

                else:
                    raise NotImplementedError("This initialization technique is not implemented (yet).")

                X0_std = (X0_tensor - col_mean) / col_std
                start = time.time()
                Xhats, stopped_early, n_steps, eta_halved, final_eta = sample_flowgem(X0_std, X0_std, M_tensor, T=param_vals["T"], eta=param_vals["eta"], sigma_mul_heur=param_vals["sigma_mul"], grad_tol=0.01)
                runtime = time.time() - start
                Xhat = torch.tensor(Xhats[-1], dtype=torch.float64) * col_std + col_mean

                imp_save = {"Xhat": Xhat, "runtime": runtime,
                            "stopped_early": stopped_early, "n_steps": n_steps,
                            "eta_halved_at": eta_halved, "final_eta": final_eta}

                torch.save(imp_save, f"{impute_dir}/imputation_{this_id}_{dataset}_{run}.pt")

            #============= EVALUATION ==============#
            Xhat = imp_save["Xhat"]
            Xstar_tensor = torch.tensor(Xstar_df.values, dtype=torch.float64)
            metric_save = {"runtime": runtime,
                        "stopped_early": stopped_early, "n_steps": n_steps,
                        "eta_halved_at": eta_halved, "final_eta": final_eta,
                        "energy_std": energy_distance_faster(Xstar_tensor, Xhat, scale=True),
                        "W2": W2(Xstar_tensor, Xhat, scale=False),
                        "MMD": MMD(Xstar_tensor, Xhat, scale=False)}

            print(metric_save)
            torch.save(metric_save, f"{metric_dir}/metrics_{this_id}_{dataset}_{run}.pt")

        #===================#
        #=== AGGREGATION ===#
        #===================#
        rows = []
        for run in range(n_runs):
            metric_save = torch.load(f"{metric_dir}/metrics_{this_id}_{dataset}_{run}.pt", weights_only=False)
            rows.append(metric_save)
        df = pd.DataFrame(rows)

        agg = {"stopped_early_frac": float(df["stopped_early"].mean()),
                "n_steps_mean": float(df["n_steps"].mean()),
                "n_steps_std": float(df["n_steps"].std(ddof=1)),
                "final_eta_mean": float(df["final_eta"].mean()),
                "final_eta_std": float(df["final_eta"].std(ddof=1)),
                "runtime_mean": float(df["runtime"].mean()),
                "runtime_std": float(df["runtime"].std(ddof=1)),
                # metrics
                "energy.std_mean": float(df["energy_std"].mean()),
                "energy.std_std": float(df["energy_std"].std(ddof=1)),
                "W2_mean": float(df["W2"].mean()),
                "W2_std": float(df["W2"].std(ddof=1)),
                "MMD_mean": float(df["MMD"].mean()),
                "MMD_std": float(df["MMD"].std(ddof=1))}

        torch.save(agg, f"{save_dir}/results_{this_id}_{dataset}.pt")



## Analyze the results

In [ ]:
import os
import numpy as np
import pandas as pd
import torch

# -----------------------
# User settings (keep your values)
# -----------------------
init_fixed = defaults["init"]
# datasets = ["windspeed"]

# Sweep settings
eta_fixed_for_sigma_sweep = defaults["eta"]
sigma_values_for_sweep = sorted(pd.Series(sigma_mul_values))

sigma_fixed_for_eta_sweep = defaults["sigma_mul"]
eta_values_for_sweep = sorted(pd.Series(eta_values))

# -----------------------
# Load parameter log
# -----------------------
df = pd.read_csv(log_file)

# Recommended dtypes
df["ID"] = df["ID"].astype(int)
df["init"] = df["init"].astype(str)
df["sigma_mul"] = df["sigma_mul"].astype(float)
df["eta"] = df["eta"].astype(float)
df["T"] = df["T"].astype(int)

# Float matching tolerance (for robust selection)
USE_TOLERANCE_MATCH = True
TOL = 1e-10

def match_value(series, value):
    if not USE_TOLERANCE_MATCH:
        return series == value
    return np.isclose(series.to_numpy(), value, atol=TOL, rtol=0)

# -----------------------
# Select configs
# -----------------------
def select_configs_sigma_sweep(init_fixed, eta_fixed, sigma_list):
    sub = df[
        (df["init"] == init_fixed) &
        (match_value(df["eta"], eta_fixed)) &
        (df["sigma_mul"].isin(sigma_list)) &
        (df["T"] == defaults["T"])
    ].copy()
    sub = sub.sort_values("sigma_mul")
    return sub

def select_configs_eta_sweep(init_fixed, sigma_fixed, eta_list):
    sub = df[
        (df["init"] == init_fixed) &
        (match_value(df["sigma_mul"], sigma_fixed)) &
        (df["eta"].isin(eta_list))&
        (df["T"] == fixed_T)
    ].copy()
    sub = sub.sort_values("eta")
    return sub

configs_sigma = select_configs_sigma_sweep(init_fixed, eta_fixed_for_sigma_sweep, sigma_values_for_sweep)
configs_eta = select_configs_eta_sweep(init_fixed, sigma_fixed_for_eta_sweep, eta_values_for_sweep)

print("Sigma sweep configs found in log:\n", configs_sigma[["ID","sigma_mul","eta"]])
print("Eta sweep configs found in log:\n", configs_eta[["ID","sigma_mul","eta"]])
print("If results are missing, they will be skipped automatically.")

# -----------------------
# Load results (skip if missing)
# -----------------------
def results_path(ID, dataset):
    return f"{save_dir}/results_{ID}_{dataset}.pt"

def load_metric_for_config(ID, dataset, metric_key_mean="energy.std_mean", metric_key_std="energy.std_std"):
    path = results_path(ID, dataset)
    if not os.path.exists(path):
        return None  # indicates missing
    obj = torch.load(path, map_location="cpu")
    mean = float(obj[metric_key_mean])
    std = float(obj[metric_key_std])
    return mean, std

def format_cell(mean, std, decimals_mean=6, decimals_std=6):
    return f"{mean:.{decimals_mean}f} ({std:.{decimals_std}f})"

# -----------------------
# Build LaTeX tables with only available configurations
# Requires \usepackage{booktabs} in your preamble.
# -----------------------

def latex_escape(s):
    """Escape characters that are special in LaTeX (for plain text, not math)."""
    replacements = {
        "\\": r"\textbackslash{}",
        "&": r"\&",
        "%": r"\%",
        "$": r"\$",
        "#": r"\#",
        "_": r"\_",
        "{": r"\{",
        "}": r"\}",
        "~": r"\textasciitilde{}",
        "^": r"\textasciicircum{}",
    }
    return "".join(replacements.get(ch, ch) for ch in str(s))


def latex_cell(text):
    """Make the output of format_cell safe for LaTeX (handles a +/- sign if present)."""
    text = str(text)
    text = text.replace("±", r"$\pm$")
    return text


def _get_reference_means(ref_ID, datasets, metric_key_mean, metric_key_std, ref_desc):
    """
    Return {dataset: reference mean} used as the divisor when standardizing.
    Raises ValueError if the reference configuration is missing for any dataset
    or if a reference mean is zero (division would be meaningless).
    """
    ref_means = {}
    for ds in datasets:
        res = load_metric_for_config(ref_ID, ds, metric_key_mean, metric_key_std)
        if res is None:
            raise ValueError(f"Cannot standardize: no results for {ref_desc} (ID={ref_ID}) on dataset '{ds}'.")
        ref_mean = res[0]
        if ref_mean == 0:
            raise ValueError(f"Cannot standardize: reference mean for {ref_desc} on dataset '{ds}' is zero.")
        ref_means[ds] = ref_mean
    return ref_means


def _build_latex_table(row_labels, column_headers, cell_fn, caption, label=None):
    """
    row_labels:     list of dataset names
    column_headers: list of already-LaTeX-formatted header strings
    cell_fn:        function(ds, col_index) -> LaTeX-safe cell string
    """
    ncols = len(column_headers)
    colspec = "l" + "c" * ncols

    lines = []
    lines.append(r"\begin{table}[htbp]")
    lines.append(r"\centering")
    lines.append(rf"\caption{{{caption}}}")
    if label:
        lines.append(rf"\label{{{label}}}")
    lines.append(rf"\begin{{tabular}}{{{colspec}}}")
    lines.append(r"\toprule")
    lines.append("Dataset & " + " & ".join(column_headers) + r" \\")
    lines.append(r"\midrule")
    for ds in row_labels:
        cells = [cell_fn(ds, j) for j in range(ncols)]
        lines.append(latex_escape(ds) + " & " + " & ".join(cells) + r" \\")
    lines.append(r"\bottomrule")
    lines.append(r"\end{tabular}")
    lines.append(r"\end{table}")
    return "\n".join(lines)


def build_latex_table_sigma(configs_sigma, datasets, metric_key_mean, metric_key_std,
                            sigma_values_for_sweep, eta_fixed_for_sigma_sweep,
                            decimals_mean=2, decimals_std=2, label=None,
                            standardize=False, reference_sigma=None):
    """
    If standardize=True, every mean and std in a row (dataset) is divided by the
    mean obtained for that dataset at sigma_mul == reference_sigma.
    """
    # Map sigma_mul -> row (if duplicates exist, keep first but warn)
    sigma_to_row = {}
    for _, r in configs_sigma.iterrows():
        sig = float(r["sigma_mul"])
        if sig not in sigma_to_row:
            sigma_to_row[sig] = r
        else:
            print(f"WARNING: duplicate sigma_mul={sig} in sigma configs; using the first ID={sigma_to_row[sig]['ID']}.")

    # Reference values for standardization (per dataset)
    ref_means = None
    if standardize:
        if reference_sigma is None:
            raise ValueError("standardize=True requires reference_sigma to be set.")
        reference_sigma = float(reference_sigma)
        if reference_sigma not in sigma_to_row:
            raise ValueError(f"reference_sigma={reference_sigma:g} not found in sigma configs.")
        ref_ID = int(sigma_to_row[reference_sigma]["ID"])
        ref_means = _get_reference_means(ref_ID, datasets, metric_key_mean, metric_key_std,
                                         f"sigma_mul={reference_sigma:g}")

    # Determine which sigma columns are available for ALL datasets
    available_sigmas = []
    for sig in sigma_values_for_sweep:
        if sig not in sigma_to_row:
            continue
        ID = int(sigma_to_row[sig]["ID"])
        ok_all = True
        for ds in datasets:
            if load_metric_for_config(ID, ds, metric_key_mean, metric_key_std) is None:
                ok_all = False
                break
        if ok_all:
            available_sigmas.append(sig)

    if len(available_sigmas) == 0:
        return f"% Sigma sweep table: no complete results available for any sigma in {sigma_values_for_sweep}."

    available_sigmas = [float(x) for x in available_sigmas]

    headers = [rf"${v:g}\hat{{\sigma}}$" for v in available_sigmas]

    def cell_fn(ds, j):
        sig = available_sigmas[j]
        ID = int(sigma_to_row[sig]["ID"])
        mean, std = load_metric_for_config(ID, ds, metric_key_mean, metric_key_std)
        if ref_means is not None:
            mean, std = mean / ref_means[ds], std / ref_means[ds]
        return latex_cell(format_cell(mean, std, decimals_mean, decimals_std))

    caption = (rf"Sigma sweep (init fixed: \texttt{{{latex_escape(init_fixed)}}}, "
               rf"$\eta$ fixed: ${eta_fixed_for_sigma_sweep:g}$")
    if standardize:
        caption += rf"; values divided by the result at ${reference_sigma:g}\hat{{\sigma}}$"
    caption += ")"
    return _build_latex_table(datasets, headers, cell_fn, caption, label)


def build_latex_table_eta(configs_eta, datasets, metric_key_mean, metric_key_std,
                          eta_values_for_sweep, sigma_fixed_for_eta_sweep,
                          decimals_mean=2, decimals_std=2, label=None,
                          standardize=False, reference_eta=None):
    """
    If standardize=True, every mean and std in a row (dataset) is divided by the
    mean obtained for that dataset at eta == reference_eta.
    """
    # Map eta -> row (if duplicates exist, keep first but warn)
    eta_to_row = {}
    for _, r in configs_eta.iterrows():
        et = float(r["eta"])
        if et not in eta_to_row:
            eta_to_row[et] = r
        else:
            print(f"WARNING: duplicate eta={et} in eta configs; using the first ID={eta_to_row[et]['ID']}.")

    # Reference values for standardization (per dataset)
    ref_means = None
    if standardize:
        if reference_eta is None:
            raise ValueError("standardize=True requires reference_eta to be set.")
        reference_eta = float(reference_eta)
        if reference_eta not in eta_to_row:
            raise ValueError(f"reference_eta={reference_eta:g} not found in eta configs.")
        ref_ID = int(eta_to_row[reference_eta]["ID"])
        ref_means = _get_reference_means(ref_ID, datasets, metric_key_mean, metric_key_std,
                                         f"eta={reference_eta:g}")

    # Determine which eta columns are available for ALL datasets
    available_etas = []
    for et in eta_values_for_sweep:
        if et not in eta_to_row:
            continue
        ID = int(eta_to_row[et]["ID"])
        ok_all = True
        for ds in datasets:
            if load_metric_for_config(ID, ds, metric_key_mean, metric_key_std) is None:
                ok_all = False
                break
        if ok_all:
            available_etas.append(et)

    if len(available_etas) == 0:
        return f"% Eta sweep table: no complete results available for any eta in {eta_values_for_sweep}."

    available_etas = [float(x) for x in available_etas]

    headers = [rf"$\eta = {v:g}$" for v in available_etas]

    def cell_fn(ds, j):
        et = available_etas[j]
        ID = int(eta_to_row[et]["ID"])
        mean, std = load_metric_for_config(ID, ds, metric_key_mean, metric_key_std)
        if ref_means is not None:
            mean, std = mean / ref_means[ds], std / ref_means[ds]
        return latex_cell(format_cell(mean, std, decimals_mean, decimals_std))

    caption = (rf"Eta sweep (init fixed: \texttt{{{latex_escape(init_fixed)}}}, "
               rf"$\sigma$ multiplier fixed: ${sigma_fixed_for_eta_sweep:g}$")
    if standardize:
        caption += rf"; values divided by the result at $\eta = {reference_eta:g}$"
    caption += ")"
    return _build_latex_table(datasets, headers, cell_fn, caption, label)

# -----------------------
# Metric keys and output
# -----------------------
metric_key_mean = "energy.std_mean"
metric_key_std = "energy.std_std"
# metric_key_mean = "n_steps_mean"
# metric_key_std = "final_eta_mean"

latex_sigma = build_latex_table_sigma(
    configs_sigma=configs_sigma,
    datasets=datasets,
    metric_key_mean=metric_key_mean,
    metric_key_std=metric_key_std,
    sigma_values_for_sweep=sigma_values_for_sweep,
    eta_fixed_for_sigma_sweep=eta_fixed_for_sigma_sweep,
    standardize=True, reference_sigma=1
)

latex_eta = build_latex_table_eta(
    configs_eta=configs_eta,
    datasets=datasets,
    metric_key_mean=metric_key_mean,
    metric_key_std=metric_key_std,
    eta_values_for_sweep=eta_values_for_sweep,
    sigma_fixed_for_eta_sweep=sigma_fixed_for_eta_sweep,
    standardize=True, reference_eta=0.01
)

print("\n--- SIGMA SWEEP TABLE (Latex) ---\n")
print(latex_sigma)
print("\n--- ETA SWEEP TABLE (Latex) ---\n")
print(latex_eta)